# Course 3: Large Language Models
## Session 6: Post-training II: learning from preferences with DPO

**Goal:** write the DPO loss, train on a small preference set, and measure both what improved and what drifted.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
import random, itertools
SUBJ = ["মা", "বাবা", "শিক্ষক", "ছেলেটি", "মেয়েটি", "কৃষক", "দাদু", "রহিম"]
TIME = ["সকালে", "বিকেলে", "রাতে", "প্রতিদিন", "আজ"]
PLACE = ["বাজারে", "স্কুলে", "মাঠে", "বাড়িতে", "গ্রামে"]
OBJ_VERB = [("ভাত", "খায়"), ("বই", "পড়ে"), ("গান", "গায়"), ("ফুটবল", "খেলে"), ("চিঠি", "লেখে"), ("মাছ", "কেনে"), ("গল্প", "বলে"), ("ছবি", "আঁকে")]
VERB_OF = dict(OBJ_VERB)
# A toy language: subject, time, place, object, verb, full stop. The verb must fit the object (ভাত খায়, বই পড়ে).
ALL_SENTENCES = [[s, t, p, o, v, "।"] for s, t, p, (o, v) in itertools.product(SUBJ, TIME, PLACE, OBJ_VERB)]

def is_valid(words):
    return (len(words) == 6 and words[0] in SUBJ and words[1] in TIME and words[2] in PLACE
            and VERB_OF.get(words[3]) == words[4] and words[5] == "।")

EXTRA = ["<pad>", "<mask>", "প্রশ্ন", "উত্তর", ":", "?", "কী", "করে"]
VOCAB = EXTRA + SUBJ + TIME + PLACE + [o for o, _ in OBJ_VERB] + [v for _, v in OBJ_VERB] + ["।"]
STOI = {w: i for i, w in enumerate(VOCAB)}
enc = lambda words: [STOI[w] for w in words]
dec = lambda ids: [VOCAB[i] for i in ids]

In [ ]:
random.seed(0); torch.manual_seed(0)
_shuffled = ALL_SENTENCES[:]; random.shuffle(_shuffled)
train_sents, val_sents = _shuffled[:1200], _shuffled[1200:]            # 400 sentences the model never sees
train_ids = torch.tensor([i for s in train_sents for i in enc(s)])
val_ids = torch.tensor([i for s in val_sents for i in enc(s)])

def get_batch(data, block_size, batch_size):
    ix = torch.randint(0, (len(data) - block_size - 1) // 6, (batch_size,)) * 6      # start on a sentence boundary
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

def train_lm(cfg, steps=600, lr=3e-3, data=None, seed=0):
    torch.manual_seed(seed); data = train_ids if data is None else data
    model = GPT(cfg); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = get_batch(data, cfg.block_size, 32)
        _, loss = model(x, y); opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def val_loss(model, block_size=24):
    n = (len(val_ids) - 1) // block_size * block_size
    x = val_ids[:n].view(-1, block_size); y = val_ids[1:n + 1].view(-1, block_size)
    return model(x, y)[1].item()

@torch.no_grad()
def sample_sentences(model, n=200, temperature=1.0):
    prompt = torch.tensor([enc(train_sents[i]) for i in torch.randint(len(train_sents), (n,)).tolist()])   # one real sentence as the prompt
    out = model.generate(prompt, 6, temperature=temperature, top_p=1.0)[:, 6:]                     # the model writes the next one
    return [dec(row.tolist()) for row in out]

In [ ]:
cfg = Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=64)
base = train_lm(cfg)
print("base model trained | validation loss %.3f | valid sentences in 200 samples: %.2f" % (val_loss(base), sum(map(is_valid, sample_sentences(base))) / 200))

### 1. Preference data
Each item is a prompt, a chosen answer and a rejected answer. Here the 'annotators' prefer sentences set in a school (স্কুলে) over the same sentence set anywhere else. Nothing tells the model the rule; it only sees pairs.

In [ ]:
import copy
random.seed(1)
pairs = []
for _ in range(300):
    s, t, (o, v) = random.choice(SUBJ), random.choice(TIME), random.choice(OBJ_VERB)
    other = random.choice([p for p in PLACE if p != "স্কুলে"])
    prefix = random.choice(train_sents)                               # a sentence of context, the same for both answers
    pairs.append((enc(prefix + [s, t, "স্কুলে", o, v, "।"]), enc(prefix + [s, t, other, o, v, "।"])))
chosen = torch.tensor([p[0] for p in pairs]); rejected = torch.tensor([p[1] for p in pairs])
print("chosen  :", " ".join(dec(pairs[0][0][6:]))); print("rejected:", " ".join(dec(pairs[0][1][6:])))

### 2. The DPO loss
No reward model and no sampling. For each pair, raise the log-probability of the chosen answer relative to a frozen reference model, and lower that of the rejected one:

`loss = -log sigmoid( beta * [ (log p(chosen) - log p_ref(chosen)) - (log p(rejected) - log p_ref(rejected)) ] )`

In [ ]:
def seq_logprob(model, seqs):
    logits, _ = model(seqs[:, :-1])
    return torch.log_softmax(logits, -1).gather(-1, seqs[:, 1:, None]).squeeze(-1).sum(1)      # log p of the whole sentence

def dpo_loss(policy, reference, chosen, rejected, beta):
    with torch.no_grad(): ref_c, ref_r = seq_logprob(reference, chosen), seq_logprob(reference, rejected)
    margin = (seq_logprob(policy, chosen) - ref_c) - (seq_logprob(policy, rejected) - ref_r)
    return -F.logsigmoid(beta * margin).mean()

print("loss at the start (policy = reference): %.4f  (it must equal log 2 = 0.6931)" % dpo_loss(base, base, chosen[:8], rejected[:8], 0.5).item())

### 3. Train, and measure three things
1. Did the preference take hold? (share of generated sentences set in a school)
2. Is the output still correct? (share of valid sentences)
3. How far did the model move from where it started? (loss on ordinary text; some rise is expected, because the model now prefers one place)

In [ ]:
def school_share(model, n=400):
    s = sample_sentences(model, n); return sum(w[2] == "স্কুলে" for w in s if len(w) > 2) / n, sum(map(is_valid, s)) / n

def run_dpo(beta, anchor=0.0, steps=150, lr=1e-3):
    torch.manual_seed(0); policy = copy.deepcopy(base); opt = torch.optim.AdamW(policy.parameters(), lr=lr)
    for _ in range(steps):
        i = torch.randint(len(chosen), (32,))
        loss = dpo_loss(policy, base, chosen[i], rejected[i], beta)
        if anchor > 0:                                                  # keep modelling ordinary text while learning the preference
            x, y = get_batch(train_ids, 24, 32); loss = loss + anchor * policy(x, y)[1]
        opt.zero_grad(); loss.backward(); opt.step()
    return policy

torch.manual_seed(0)
print("model                          school share   valid   loss on ordinary text")
s0, v0 = school_share(base); print(f"base                           {s0:12.2f}   {v0:5.2f}   {val_loss(base):.3f}")
rows = {}
for name, beta, anchor in [("DPO alone, beta 0.5", 0.5, 0.0), ("DPO + anchor, beta 0.1", 0.1, 1.0), ("DPO + anchor, beta 0.5", 0.5, 1.0), ("DPO + anchor, beta 2.0", 2.0, 1.0)]:
    p = run_dpo(beta, anchor); sh, v = school_share(p); rows[name] = (sh, v, val_loss(p))
    print(f"{name:30s} {sh:12.2f}   {v:5.2f}   {rows[name][2]:.3f}")
assert rows["DPO alone, beta 0.5"][1] < 0.6 and rows["DPO + anchor, beta 0.5"][0] > s0 + 0.5 and rows["DPO + anchor, beta 0.5"][1] > 0.9

Two lessons.

**DPO alone broke the model.** The loss can be lowered in two ways: make the chosen answer more likely, or make the rejected answer less likely. The second is easy to overdo: the model learns to write nonsense after any place that is not a school, which drives the rejected answers' probability towards zero and wrecks ordinary sentences. This failure is known in the DPO literature, where the likelihood of both answers can fall during training.

**Anchoring fixes it.** Adding the ordinary language-modelling loss on normal text keeps the model fluent while the preference is learned. Real pipelines get the same effect from a small learning rate, few steps, LoRA adapters, or a mixed-in supervised loss. `beta` then sets how far the policy may move: compare the school share at 0.5 and at 2.0.

### 4. What the model was never told
The pairs said nothing about subjects, objects or times. Check that DPO did not change those by accident.

In [ ]:
from collections import Counter
p = run_dpo(0.5, anchor=1.0); torch.manual_seed(0)
before, after = Counter(w[0] for w in sample_sentences(base, 800)), Counter(w[0] for w in sample_sentences(p, 800))
print("subject      before   after")
for s in SUBJ: print(f"{s:10s} {before[s] / 800:8.2f} {after[s] / 800:7.2f}")

### 5. DPO on a real model (NOT RUN here)

In [ ]:
# NOT RUN: needs a GPU and `pip install transformers peft trl datasets`.
# 1. Start from your session 5 model (the base model plus the trained LoRA adapter).
# 2. Build 500 preference pairs in Bangla: a prompt, a chosen answer and a rejected answer.
#    Cheap way to start: generate two answers per prompt at temperature 1.0 and pick the better one by hand with a written rule.
# 3. Train with trl's DPOTrainer (dataset columns: "prompt", "chosen", "rejected"); start with beta = 0.1.
# 4. Evaluate blind: show yourself 50 pairs of (session 5 answer, DPO answer) in random order and count wins.

### TODO
1. Make the preferences noisy: flip 30% of the pairs. How much of the effect survives?
2. For 'DPO alone', print 10 generated sentences and log the mean log-probability of the chosen and of the rejected answers during training. Which one moves more?
3. Explain in four sentences how DPO relates to RLHF with a reward model and PPO.